## Creating a view for dimension

In [0]:
%sql
CREATE OR REPLACE VIEW transport_lakehouse.gold.dim_manufacturer AS
WITH base AS (
  SELECT
    CAST(manufacturer_cd AS STRING) AS manufacturer_cd,
    TRIM(manufacturer_nm) AS manufacturer_nm
  FROM (
    SELECT manufacturer_cd, manufacturer_nm
    FROM transport_lakehouse.silver.silver_vehicles_two_wheeled
    UNION ALL
    SELECT manufacturer_cd, manufacturer_nm
    FROM transport_lakehouse.silver.silver_vehicles_public
    UNION ALL
    SELECT manufacturer_cd, manufacturer_nm
    FROM transport_lakehouse.silver.silver_vehicles_private
  )
),
dedup AS (
  SELECT
    manufacturer_cd,
    MAX(manufacturer_nm) AS manufacturer_nm
  FROM base
  WHERE manufacturer_cd IS NOT NULL
  GROUP BY manufacturer_cd
)
SELECT
  ROW_NUMBER() OVER (ORDER BY manufacturer_cd) AS manufacturer_key,
  manufacturer_cd,
  manufacturer_nm
FROM dedup;


## Quality check

In [0]:
%sql
select *
from transport_lakehouse.gold.dim_manufacturer

In [0]:
%sql
SELECT manufacturer_cd, COUNT(*) AS cnt
FROM transport_lakehouse.gold.dim_manufacturer
GROUP BY manufacturer_cd
HAVING COUNT(*) > 1
ORDER BY cnt DESC;
